# 📝 그래프 데이터 과학 과제 LV1(기초): 투영·PageRank·개인화 PageRank

> 이 단원의 새 기술을 **하나씩** 확인합니다. 문제는 네 갈래로 묶여 있습니다.
>
> - **1. 투영 만들기와 레이블 고르기**: `gds.graph.project`·레이블을 하나만 담았을 때의 관계 0건
> - **2. 투영 관리와 검산**: `gds.graph.list`·무방향 투영의 관계 수 2배 검산
> - **3. 중심성과 실행 모드**: `gds.pageRank.stream`·`gds.degree.stream`·`write` 저장·저장한 속성 조회
> - **4. 개인화 PageRank 와 투영 정리**: `sourceNodes`·`gds.graph.drop`

## 풀이 방법
1. 맨 위 **준비 셀**(연결 → 초기화(투영·그래프) → 데이터 적재)을 위에서부터 실행하세요.
2. 각 문제의 **답안 셀**을 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: 의료 지식 그래프의 **약효분류-약물 축**입니다. `PharmacologicClass`(약효분류) 노드가 `INCLUDES` 로 `Compound`(약물)를 거느리고, 약물끼리는 `RESEMBLES_CC`(화학적으로 닮음)로 이어져 있습니다.
- 반드시 **실습 전용 DB**에 연결하세요(준비 셀이 그래프를 지우고 새로 만듭니다).
- PageRank 점수는 실행마다 같은 값이지만, 채점은 **1위 이름·개수·참거짓**으로 봅니다.

화이팅!

아래 준비 셀들을 위에서부터 실행하세요.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 지우기 전에 이 DB 가 맞는지 먼저 확인합니다.
UNIT_LABELS = ['Compound', 'DayType', 'Disease', 'ExDenseEvent', 'ExDisease', 'ExDomestic', 'ExDrug', 'ExEvent', 'ExForeign', 'ExIdKey', 'ExKing', 'ExKinng', 'ExNameKey', 'ExNodeKeyDemo', 'ExPerson', 'ExReign', 'ExScopeEvent', 'ExThrone', 'ExUniqueDemo', 'ExWorld', 'ExYear', 'FlatDrug', 'FlatKing', 'FlatOrder', 'Gene', 'IdKey', 'Line', 'NameKey', 'NodeClass', 'NodeDay', 'NodeDrug', 'NodeKing', 'NodeMonth', 'NodeOrder', 'NodeYear', 'PharmacologicClass', 'Station', 'SurveyDay', 'SurveyYear', 'Symptom', 'TempStation', 'TryDay', 'TryLine', 'TryStation']   # 이 단원이 만드는 레이블 전부(앞 일차가 남긴 것까지)

# 이 단원 것이 아닌 노드가 하나라도 있으면 지우지 않고 멈춥니다.
# .env 의 주소가 어긋나도 접속은 조용히 성공하므로, 지우기 전에 확인하는 수밖에 없습니다.
# all 로 보는 이유: any 로 보면 :Person:PatientRecord 처럼 한 레이블만 겹치는 남의 노드가 통과합니다
foreign = run_cypher("""
MATCH (n) WHERE size(labels(n)) = 0
   OR NOT all(label IN labels(n) WHERE label IN $unit_labels)
RETURN DISTINCT labels(n) AS labels LIMIT 5""", unit_labels=UNIT_LABELS)
if foreign:
    raise RuntimeError(
        f"{NEO4J_URI} 에 이 단원 것이 아닌 노드가 있습니다: {foreign}\n"
        "다른 실습이나 개인 데이터가 든 DB 로 보여 초기화를 멈췄습니다.\n"
        ".env 의 NEO4J_URI 가 실습 전용 DB 를 가리키는지 먼저 확인하세요.\n"
        "주소가 맞다면 위 레이블은 앞 실습이 남긴 것입니다. UNIT_LABELS 에 더하고 다시 실행하세요.")

# 여기까지 왔으면 이 DB 에는 이 단원이 만든 노드밖에 없습니다.
# 1) 메모리에 올라온 투영부터 내린다. 투영은 이름이 겹치면 다시 못 만들어 재실행이 막힌다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName",
               name=_g["graphName"])

# 2) 저장된 그래프를 지운다. DETACH: 노드에 붙은 관계까지 함께 지운다
run_cypher("MATCH (n) DETACH DELETE n")

print("초기화 완료:", NEO4J_URI,
      "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"],
      "· 남은 투영:", len(run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")))

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# data/hetionet_*.csv 는 Hetionet v1.0 에서 재배포 가능한 출처(CC0/CC BY)만 골라 낸 조각입니다.
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
NODE_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"]
# 관계 타입마다 양끝 레이블이 정해져 있습니다. 적재할 때 이 표로 레이블을 찍어 줘야
# MATCH 가 인덱스를 타고, 그래야 9만 건이 몇 초 안에 들어갑니다.
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

# 1) 레이블마다 id 인덱스를 먼저 만든다. 관계를 붙일 때 이 인덱스로 노드를 찾는다
for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

# 2) 노드 csv 를 읽어 레이블별로 나눠 담는다. 레이블마다 CREATE 쿼리가 달라서 미리 갈라 둔다
_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    # 초기화 직후라 같은 노드가 있을 수 없다. MERGE 대신 CREATE 가 훨씬 빠르다
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

# 3) 관계 csv 도 타입별로 나눠 담는다
_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]   # 이 타입의 출발·도착 레이블을 위 표에서 꺼낸다
    # 2만 건씩 끊어 보낸다. 9만 건을 한 트랜잭션에 넣으면 메모리가 크게 뛴다
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 이번 과제가 쓰는 조각을 훑어봅니다.

In [ ]:
# [제공 코드] 이번 과제가 쓰는 두 노드 종류와 두 관계 타입을 확인합니다
counts = run_cypher("""
    MATCH (n)
    WHERE n:PharmacologicClass OR n:Compound
    RETURN labels(n)[0] AS label, count(*) AS cnt ORDER BY cnt DESC
""")
for row in counts:
    print(f"  {row['label']:20} {row['cnt']:>6,}")
rel_counts = run_cypher("""
    MATCH ()-[r]->()
    WHERE type(r) IN ['INCLUDES', 'RESEMBLES_CC']
    RETURN type(r) AS rel_type, count(*) AS cnt ORDER BY cnt DESC
""")
for row in rel_counts:
    print(f"  {row['rel_type']:20} {row['cnt']:>6,}")
# 약효분류가 어떤 약을 거느리는지 예로 한 분류를 본다(4-1 이 묻는 분류와는 다른 분류다.
# 4-1 정답을 미리 보여 주지 않으려고 일부러 다른 분류를 고른다)
sample = run_cypher("""
    MATCH (p:PharmacologicClass {name: 'Thiazides'})-[:INCLUDES]->(c:Compound)
    RETURN c.name AS drug ORDER BY drug
""")
print("Thiazides 에 속한 약:", [row["drug"] for row in sample])

---
# 1. 투영 만들기와 레이블 고르기

투영을 만들고, 레이블을 잘못 고르면 관계가 통째로 빠지는 것을 확인합니다(교안_01 2·3절).

## 1-1. 그래프 투영 만들기
**배경**: GDS 분석은 **투영(메모리 사본)** 위에서 합니다. 약효분류-약물 축을 담은 투영을 만듭니다.

**요구사항**:
- `PharmacologicClass` 와 `Compound` 노드, `INCLUDES` 와 `RESEMBLES_CC` 관계를 **`classGraph`** 라는 이름으로 투영하세요.
- 두 관계 모두 **방향을 지워서**(`orientation: 'UNDIRECTED'`) 담으세요.
- 투영 결과의 `nodeCount` 를 변수 **`n_nodes`** 에 담으세요.

**예시**: 약효분류 345개 + 약물 1,531개이므로 `n_nodes` 는 **1,876** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 투영을 만드는 프로시저는 gds.graph.project 다. 이름·노드 레이블·관계 설정 세 가지를 넘긴다.
- 방향을 지우려면 관계 자리에 리스트가 아니라 설정 맵을 넣는다.

세부구현:
1. 노드 레이블은 두 개짜리 리스트로 넘긴다.
2. 관계 자리에는 타입 이름을 키로 하고 orientation 설정을 값으로 갖는 맵을 넘긴다.
   두 타입 모두 같은 설정을 적어 준다.
3. YIELD 로 nodeCount 만 골라 받고, 첫 행에서 그 값을 꺼내 n_nodes 에 담는다.
```

</details>

In [ ]:
res = run_cypher('''
CALL gds.graph.project(
    'classGraph',
    ['PharmacologicClass', 'Compound'],
    {
        INCLUDES: {type: 'INCLUDES', orientation: 'UNDIRECTED'},
        RESEMBLES_CC: {type: 'RESEMBLES_CC', orientation: 'UNDIRECTED'}
    }
)
YIELD nodeCount
RETURN nodeCount
''')
n_nodes = res[0]['nodeCount']
print(f'n_nodes: {n_nodes}')


In [ ]:
# [자가채점]
assert n_nodes == 1876, '투영에 담긴 노드 수가 다릅니다. 레이블을 PharmacologicClass 와 Compound 두 개로 줬는지, YIELD 로 nodeCount 를 받았는지 확인하세요'
# 값만 적어도 통과하지 않게, 투영이 실제로 메모리에 있는지 카탈로그에서 확인한다
_made = run_cypher("CALL gds.graph.list('classGraph') YIELD nodeCount "
                   "RETURN nodeCount")
assert _made, 'classGraph 투영이 없습니다. 투영을 실제로 만들었는지 확인하세요'
assert _made[0]['nodeCount'] == n_nodes, \
    '카탈로그의 노드 수와 담은 값이 다릅니다. YIELD 로 받은 값을 그대로 담았는지 확인하세요'
print('✅ 통과!')

## 1-2. 레이블을 하나만 담으면 어떻게 되나
**배경**: 관계는 **양쪽 끝 노드가 모두 투영에 담겨야** 따라옵니다. 직접 확인해 봅니다.

**요구사항**:
- `PharmacologicClass` **하나만** 담고 `INCLUDES` 관계를 넣어 **`onlyClassGraph`** 를 만드세요 (방향 설정은 하지 않아도 됩니다).
- 투영 결과의 `relationshipCount` 를 변수 **`only_rels`** 에 담으세요.

**예시**: 노드는 345개 담기지만 `only_rels` 는 **0** 입니다. `INCLUDES` 의 도착점인 `Compound` 가 투영에 없기 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 1-1 과 같은 프로시저인데 레이블만 하나로 줄인다.

세부구현:
1. 노드 레이블 자리에 약효분류 하나만 넣는다(리스트 한 개짜리도 되고 문자열 하나도 된다).
2. 관계는 타입 이름 하나만 리스트로 넘긴다.
3. YIELD 로 relationshipCount 를 받아 only_rels 에 담고 출력한다.
```

</details>

In [ ]:
res = run_cypher('''
CALL gds.graph.project(
    'onlyClassGraph',
    'PharmacologicClass',
    {
        INCLUDES: {type: 'INCLUDES', orientation: 'UNDIRECTED'}
    }
)
YIELD relationshipCount
RETURN relationshipCount
''')
only_rels = res[0]['relationshipCount']
print(f'only_rels: {only_rels}')


In [ ]:
# [자가채점]
assert only_rels == 0, 'INCLUDES 의 도착점(Compound)을 빼면 관계가 0 건이어야 합니다. 레이블에 Compound 를 함께 넣지 않았는지 확인하세요'
# 값만 적어도 통과하지 않게, 투영이 실제로 만들어졌는지 카탈로그에서 확인한다
_only = run_cypher("CALL gds.graph.list('onlyClassGraph') "
                   "YIELD nodeCount, relationshipCount "
                   "RETURN nodeCount, relationshipCount")
assert _only, 'onlyClassGraph 투영이 없습니다. 투영을 실제로 만들었는지 확인하세요'
assert _only[0]['nodeCount'] == 345, \
    f'onlyClassGraph 에 노드가 {_only[0]["nodeCount"]}개 담겼습니다. 약효분류 345개만 담았는지 확인하세요'
assert _only[0]['relationshipCount'] == 0, \
    f'onlyClassGraph 의 관계가 {_only[0]["relationshipCount"]}건입니다. '\
    '레이블 하나만 담았는지 확인하세요'
print('✅ 통과!')

---
# 2. 투영 관리와 검산

만든 투영을 목록으로 확인하고, 무방향으로 담겼는지 관계 수로 검산합니다(교안_01 3·4절).

## 2-1. 투영 목록에서 이름 확인하기
**배경**: 지금 메모리에 어떤 투영이 올라와 있는지 목록으로 확인합니다.

**요구사항**:
- `gds.graph.list()` 로 투영 이름들을 모아 리스트 **`graph_names`** 에 담으세요.
- 이름과 함께 `memoryUsage` 도 YIELD 로 받아 이름·메모리를 나란히 출력하세요(채점은 이름 리스트만 봅니다).

**예시**: `graph_names` 에는 `'classGraph'` 와 `'onlyClassGraph'` 가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 지금 메모리에 올라온 투영을 보여 주는 프로시저는 gds.graph.list 다.
  이름 항목과 메모리 항목 두 개를 골라 받는다.

세부구현:
1. 목록 프로시저를 호출하고, 돌려주는 값 중 투영 이름(graphName)과
   메모리 사용량(memoryUsage)을 YIELD 로 고른다.
2. 받은 dict 리스트를 한 번 돌며 이름과 메모리를 나란히 출력한다.
3. 같은 리스트에서 이름 키만 뽑아 파이썬 리스트 graph_names 로 만든다.
```

</details>

In [ ]:
res = run_cypher('CALL gds.graph.list() YIELD graphName RETURN graphName')
graph_names = [row['graphName'] for row in res]
print(f'graph_names: {graph_names}')


In [ ]:
# [자가채점]
assert isinstance(graph_names, list), 'graph_names 는 이름 문자열들의 리스트여야 합니다. 각 행에서 graphName 만 뽑았는지 확인하세요'
assert 'classGraph' in graph_names, '목록에 classGraph 가 없습니다. 1-1 투영 셀을 먼저 실행했는지 확인하세요'
assert 'onlyClassGraph' in graph_names, '목록에 onlyClassGraph 가 없습니다. 1-2 투영 셀을 먼저 실행했는지 확인하세요'
# 이름을 손으로 적어도 통과하지 않게, 지금 카탈로그를 다시 읽어 집합으로 대조한다
_live = {r['graphName'] for r in run_cypher(
    "CALL gds.graph.list() YIELD graphName RETURN graphName")}
assert set(graph_names) == _live, \
    '2-1 답안 셀을 방금 다시 실행했는지 먼저 확인하세요(뒤 문항에서 투영을 만들거나 내린 뒤 채점 셀만 다시 돌리면 목록이 달라집니다). graph_names 가 지금 메모리에 올라온 투영 목록과 다릅니다. gds.graph.list() 결과를 그대로 담았는지 확인하세요'
print('✅ 통과!')

## 2-2. 무방향으로 담겼는지 검산하기
**배경**: 무방향 투영은 관계 하나를 **양쪽 두 건**으로 담습니다. 그래서 원본을 Cypher 로 센 건수와 비교하면 제대로 담겼는지 알 수 있습니다.

**요구사항**:
- Cypher 로 `INCLUDES` 와 `RESEMBLES_CC` 관계 건수를 세어 변수 **`raw_rels`** 에 담으세요 (정답은 7,515).
- `gds.graph.list('classGraph')` 의 `relationshipCount` 를 변수 **`proj_rels`** 에 담으세요.
- `proj_rels` 가 `raw_rels` 의 2배인지 판정해 변수 **`is_undirected`** 에 참거짓으로 담으세요.

**예시**: `raw_rels` 는 7,515, `proj_rels` 는 15,030, `is_undirected` 는 `True` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 원본 건수는 평범한 MATCH 로 센다. 관계 타입 두 개를 한 번에 세려면 WHERE 로 타입을 걸러도 되고
  패턴에 파이프 기호로 두 타입을 나열해도 된다.
- 투영 쪽 건수는 목록 프로시저에서 관계 수만 골라 받는다.

세부구현:
1. 원본 관계를 세어 raw_rels 에 담는다.
2. 투영 이름을 인자로 준 목록 프로시저에서 관계 수를 받아 proj_rels 에 담는다.
3. 두 값을 비교하는 식을 그대로 is_undirected 에 담는다(비교식의 결과가 곧 참거짓이다).
```

</details>

In [ ]:
raw_rels = run_cypher('MATCH ()-[r]->() WHERE type(r) IN ["INCLUDES", "RESEMBLES_CC"] RETURN count(r) AS cnt')[0]['cnt']
proj_rels = run_cypher("CALL gds.graph.list('classGraph') YIELD relationshipCount RETURN relationshipCount")[0]['relationshipCount']
is_undirected = (proj_rels == raw_rels * 2)
print(f'raw_rels: {raw_rels}, proj_rels: {proj_rels}, is_undirected: {is_undirected}')


In [ ]:
# [자가채점]
assert raw_rels == 7515, 'Cypher 로 센 원본 관계 수가 다릅니다. INCLUDES 와 RESEMBLES_CC 두 타입만 셌는지 확인하세요'
assert proj_rels == 15030, '투영의 관계 수가 다릅니다. 1-1 에서 두 관계 모두 UNDIRECTED 로 담았는지 확인하세요. 고쳐 다시 만들려면 맨 위 제공 코드 셀 셋(연결·초기화·적재)을 위에서부터 다시 실행한 뒤 1-1 부터 순서대로 실행하세요'
assert is_undirected is True, '무방향이면 투영 관계 수가 원본의 2배여야 합니다. 1-1 투영을 다시 확인하세요'
# 두 값을 손으로 적어도 통과하지 않게, 채점이 그 자리에서 다시 세어 대조한다
_raw = run_cypher("MATCH ()-[r:INCLUDES|RESEMBLES_CC]->() RETURN count(r) AS c")[0]['c']
_proj = run_cypher("CALL gds.graph.list('classGraph') YIELD relationshipCount "
                   "RETURN relationshipCount AS c")[0]['c']
assert raw_rels == _raw, '원본 관계 수를 직접 세었는지 확인하세요(지금 다시 센 값과 다릅니다)'
assert proj_rels == _proj, '투영 관계 수를 카탈로그에서 받았는지 확인하세요(지금 다시 읽은 값과 다릅니다)'
print('✅ 통과!')

---
# 3. 중심성과 실행 모드

같은 투영을 두 잣대로 재어 견주고, 점수를 원본에 저장해 Cypher 로 다시 조회합니다(교안_01 6절 · 교안_02 1·2절).

## 3-1. PageRank 1위 찾기
**배경**: 이 조각에서 **점수 높은 이웃과 가장 잘 이어진 노드**가 무엇인지 봅니다.

**요구사항**:
- `classGraph` 에서 `gds.pageRank.stream` 을 실행하세요.
- 점수가 가장 높은 노드의 **이름**을 변수 **`top_node`** 에 담으세요.
- 노드 이름은 `gds.util.asNode(nodeId).name` 으로 꺼냅니다.

**예시**: `top_node` 는 **`'Eltrombopag'`** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 알고리즘 호출은 투영 이름 하나만 넘기면 된다. 결과는 노드 번호와 점수 두 컬럼이다.
- 노드 번호는 내부 번호라 사람이 읽을 이름으로 바꿔야 한다.

세부구현:
1. stream 모드로 실행하고 nodeId 와 score 를 받는다.
2. RETURN 에서 노드를 꺼내 name 을 고른다.
3. 점수 내림차순으로 정렬하고 한 건만 받는다.
4. 첫 행의 이름을 top_node 에 담는다.
```

</details>

In [ ]:
res = run_cypher('''
CALL gds.pageRank.stream('classGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name
ORDER BY score DESC, name ASC
LIMIT 1
''')
top_node = res[0]['name']
print(f'top_node: {top_node}')


In [ ]:
# [자가채점]
assert top_node == 'Eltrombopag', 'PageRank 1위 이름이 다릅니다. classGraph 로 실행했는지, score 내림차순으로 정렬했는지 확인하세요'
# 지문이 답을 알려 주므로, 채점이 그 자리에서 다시 돌려 대조한다(0.1초면 끝난다)
_pr = run_cypher("CALL gds.pageRank.stream('classGraph') YIELD nodeId, score "
                 "RETURN gds.util.asNode(nodeId).name AS name "
                 "ORDER BY score DESC, name LIMIT 1")[0]['name']
assert top_node == _pr, \
    '지금 다시 계산한 1위와 다릅니다. gds.pageRank.stream 을 직접 실행했는지 확인하세요'
print('✅ 통과!')

## 3-2. 차수 1위와 비교하기
**배경**: **선의 개수**만 세는 차수로 1위를 뽑으면 PageRank 와 같은 답이 나올까요?

**요구사항**:
- 같은 투영에서 `gds.degree.stream` 을 실행해 1위 노드 이름을 변수 **`deg_top`** 에 담으세요.
- `deg_top` 과 3-1 의 `top_node` 가 **서로 다른지** 판정해 변수 **`tops_differ`** 에 참거짓으로 담으세요.

**예시**: `deg_top` 은 **`'Diphenhydramine'`**, `tops_differ` 는 `True` 입니다(두 1위가 서로 다르므로).

<details><summary>힌트</summary>

```text
접근방법:
- 3-1 과 문법이 같다. 알고리즘 이름만 바꾼다.

세부구현:
1. degree 알고리즘을 stream 으로 실행해 이름과 점수를 받는다.
2. 내림차순 정렬 후 한 건만 받아 이름을 deg_top 에 담는다.
3. 두 이름이 서로 다른지 비교한 결과를 tops_differ 에 담는다.
```

</details>

In [ ]:
res = run_cypher('''
CALL gds.degree.stream('classGraph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name
ORDER BY score DESC, name ASC
LIMIT 1
''')
deg_top = res[0]['name']
tops_differ = (deg_top != top_node)
print(f'deg_top: {deg_top}, tops_differ: {tops_differ}')


In [ ]:
# [자가채점]
assert deg_top == 'Diphenhydramine', '차수 1위 이름이 다릅니다. gds.degree.stream 을 classGraph 로 실행했는지 확인하세요'
assert tops_differ is True, '두 1위는 서로 다른 노드여야 합니다(tops_differ 는 True). top_node 를 3-1 에서 제대로 담았는지 확인하세요'
# 차수는 재기 싼 값이라 채점이 그 자리에서 한 번 더 재어 대조한다
_deg = run_cypher("CALL gds.degree.stream('classGraph') YIELD nodeId, score "
                  "RETURN gds.util.asNode(nodeId).name AS name "
                  "ORDER BY score DESC, name LIMIT 1")[0]['name']
assert deg_top == _deg, \
    '지금 다시 잰 차수 1위와 다릅니다. gds.degree.stream 을 직접 실행했는지 확인하세요'
print('✅ 통과!')

## 3-3. 방향과 관계를 골라 차수 다시 재기
**배경**: `gds.degree.stream` 은 설정 없이 부르면 **나가는 선(외차수)** 만 셉니다. `orientation` 으로 방향을, `relationshipTypes` 로 관계 종류를 고르면 **같은 투영 하나로 다른 질문**에 답할 수 있습니다.

지금까지 쓴 `classGraph` 는 무방향이라 방향을 나눌 수 없습니다. **방향을 살린 투영을 새로 하나** 만들어 비교합니다.

**요구사항**:
- `PharmacologicClass` 와 `Compound` 노드, `INCLUDES`·`RESEMBLES_CC` 관계를 **방향을 살린 채** (`orientation` 을 주지 않고) **`classDirected`** 로 투영하세요.
- 그 투영에서 `relationshipTypes: ['INCLUDES']` 로 관계를 좁혀 차수를 두 번 잽니다.
  - 기본(나가는 선)의 1위 이름을 **`out_top`** 에 담으세요.
  - `orientation: 'REVERSE'`(들어오는 선)의 1위 이름을 **`in_top`** 에 담으세요.
- 나가는 쪽 1위는 외차수 22 로 동점인 약효분류가 둘입니다. 정렬 기준에 이름 오름차순을 두 번째로 넣어 답을 하나로 정하세요.

**예시**: `out_top` 은 `'Corticosteroid Hormone Receptor Agonists'`(약효분류), `in_top` 은 `'Eltrombopag'`(약물)입니다. `INCLUDES` 는 약효분류에서 약물로 갑니다. 그래서 **나가는 쪽 1위는 가장 많은 약을 거느린 약효분류**, **들어오는 쪽 1위는 가장 많은 약효분류에 속한 약**입니다. 방향 한 줄로 답의 **종류부터** 달라집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 투영은 1-1 과 같은 프로시저인데, 관계를 설정 맵이 아니라 **이름 리스트**로 넘기면 원본 방향이 남는다.
- 차수는 두 번째 인자에 설정 맵을 준다. 두 설정을 함께 줄 수도 있다.

세부구현:
1. 1-1 과 같은 투영 프로시저를 쓰되, 관계를 설정 맵 대신 이름 리스트로 넘긴다.
   투영 이름은 classDirected 다.
2. 차수 프로시저를 두 번 부른다. 한 번은 관계 종류만 좁히고,
   한 번은 거기에 방향을 뒤집는 설정을 더한다.
3. 각각 score 내림차순·이름 오름차순으로 정렬해 1위 이름을 꺼낸다.
```

</details>

In [ ]:
run_cypher('''
CALL gds.graph.project(
    'classDirected',
    ['PharmacologicClass', 'Compound'],
    {
        INCLUDES: {type: 'INCLUDES', orientation: 'NATURAL'},
        RESEMBLES_CC: {type: 'RESEMBLES_CC', orientation: 'UNDIRECTED'}
    }
)
YIELD graphName
''')
out_top = run_cypher('''
CALL gds.degree.stream('classDirected', {relationshipTypes: ['INCLUDES'], orientation: 'NATURAL'})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name
ORDER BY score DESC, name ASC
LIMIT 1
''')[0]['name']
in_top = run_cypher('''
CALL gds.degree.stream('classDirected', {relationshipTypes: ['INCLUDES'], orientation: 'REVERSE'})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name
ORDER BY score DESC, name ASC
LIMIT 1
''')[0]['name']
print(f'out_top: {out_top}, in_top: {in_top}')


In [ ]:
# [자가채점]
assert out_top == 'Corticosteroid Hormone Receptor Agonists', \
    '나가는 쪽 1위가 다릅니다. relationshipTypes 를 INCLUDES 하나로 줬는지, 동점이 있으니 이름 오름차순을 두 번째 정렬 기준으로 넣었는지 확인하세요'
assert in_top == 'Eltrombopag', \
    '들어오는 쪽 1위가 다릅니다. orientation 을 REVERSE 로 줬는지 확인하세요'
# 방향을 살려 담았는지 확인한다. 무방향으로 담으면 두 답이 같아져 이 문항이 무의미해진다
_dir = run_cypher("CALL gds.graph.list('classDirected') YIELD relationshipCount "
                  "RETURN relationshipCount AS c")
assert _dir, 'classDirected 투영이 없습니다. 투영을 실제로 만들었는지 확인하세요'
_raw = run_cypher("MATCH ()-[r:INCLUDES|RESEMBLES_CC]->() RETURN count(r) AS c")[0]['c']
assert _dir[0]['c'] == _raw, \
    f'투영 관계가 {_dir[0]["c"]}건인데 원본은 {_raw}건입니다. 2배라면 orientation 을 주지 않았는지, 더 적다면 INCLUDES 와 RESEMBLES_CC 를 둘 다 담았는지 확인하세요'
# 이름만 적어도 통과하지 않게, 채점이 두 설정을 그 자리에서 다시 돌려 대조한다
_q = ("CALL gds.degree.stream('classDirected', $cfg) YIELD nodeId, score "
      "RETURN gds.util.asNode(nodeId).name AS name ORDER BY score DESC, name LIMIT 1")
assert out_top == run_cypher(_q, cfg={'relationshipTypes': ['INCLUDES']})[0]['name'], \
    '지금 다시 잰 값과 다릅니다. gds.degree.stream 을 직접 실행했는지 확인하세요'
assert in_top == run_cypher(
    _q, cfg={'relationshipTypes': ['INCLUDES'], 'orientation': 'REVERSE'})[0]['name'], \
    '지금 다시 잰 값과 다릅니다. orientation 을 REVERSE 로 줬는지 확인하세요'
print('✅ 통과!')

## 3-4. 점수를 저장하기: `write` 와 `mutate`
**배경**: `stream` 은 화면으로 흘려보낼 뿐입니다. 나중에 Cypher 로 조회하려면 **원본에 저장**해야 합니다.

저장에는 두 갈래가 있습니다. `write` 는 원본 데이터베이스에, `mutate` 는 투영 안에만 남깁니다. 둘을 직접 해 보고 차이를 확인합니다.

**요구사항**:
- `gds.pageRank.write` 로 `classGraph` 의 PageRank 를 **`class_rank`** 라는 속성으로 저장하세요.
- 반환값 `nodePropertiesWritten` 을 변수 **`n_written`** 에 담으세요.
- 같은 PageRank 를 `gds.pageRank.mutate` 로 **`pr_mutated`** 라는 이름으로 투영에만 남기고, 반환값 `nodePropertiesWritten` 을 **`n_mutated`** 에 담으세요.
- Cypher 로 `pr_mutated` 속성이 있는 노드 수를 세어 **`n_in_db`** 에 담으세요. **0** 이 나와야 합니다. `mutate` 는 원본에 쓰지 않습니다.

**예시**: 투영에 담긴 노드 전부에 값이 쓰이므로 `n_written` 은 **1,876**, `n_mutated` 도 **1,876** 입니다. 그런데 `n_in_db` 는 **0** 입니다.

> `mutate` 는 같은 이름을 투영에 두 번 남기지 못합니다. 이 셀을 다시 실행해야 하면 맨 위 제공 코드 셀 셋(연결·초기화·적재)을 위에서부터 다시 실행한 뒤 1-1 부터 순서대로 오세요.

<details><summary>힌트</summary>

```text
접근방법:
- stream 자리를 write 로 바꾸고, 두 번째 인자로 설정 맵을 넘긴다.
- 저장할 속성 이름은 설정 맵의 writeProperty 키에 적는다.
- mutate 는 write 와 호출 모양이 같다. 설정 맵에서 속성 이름을 적는 키 이름만 다르다
  (write 가 writeProperty 면 mutate 는 mutateProperty 다).

세부구현:
1. write 모드로 호출하며 투영 이름과 설정 맵을 넘긴다.
2. YIELD 로 nodePropertiesWritten 을 골라 받는다.
3. 첫 행에서 그 값을 꺼내 n_written 에 담는다.
4. 같은 모양으로 mutate 를 한 번 더 부르고, 속성 이름 키만 바꿔 n_mutated 에 담는다.
5. 평범한 MATCH 로 pr_mutated 를 가진 노드를 세어 n_in_db 에 담는다.
   속성 이름으로 IS NOT NULL 을 쓰면 '그 속성 이름이 없다'는 경고가 함께 뜬다.
   에러가 아니라 mutate 가 원본에 아무것도 안 썼다는 증거다.
   경고 없이 세려면 keys(n) 에 이름이 들어 있는지로 센다.
```

</details>

In [ ]:
res = run_cypher('''
CALL gds.pageRank.write('classGraph', {
    writeProperty: 'class_rank'
})
YIELD nodePropertiesWritten
RETURN nodePropertiesWritten
''')
n_written = res[0]['nodePropertiesWritten']
print(f'n_written: {n_written}')


In [ ]:
# [자가채점]
assert n_written == 1876, '속성이 쓰인 노드 수가 다릅니다. classGraph 에 write 했는지, YIELD 로 nodePropertiesWritten 을 받았는지 확인하세요'
check = run_cypher("MATCH (n) WHERE n.class_rank IS NOT NULL RETURN count(n) AS cnt")[0]['cnt']
assert check == 1876, '속성 이름이 class_rank 가 아닌 것 같습니다. writeProperty 를 확인하세요'
assert n_mutated == 1876, 'mutate 로 값이 쓰인 노드 수가 다릅니다. classGraph 에 mutate 했는지, YIELD 로 nodePropertiesWritten 을 받았는지 확인하세요'
assert n_in_db == 0, \
    'mutate 는 원본에 쓰지 않습니다. n_in_db 는 0 이어야 합니다. write 를 두 번 하지 않았는지 확인하세요'
# 값만 적어도 통과하지 않게, 투영 안과 원본을 채점이 그 자리에서 직접 들여다본다
_sch = run_cypher("CALL gds.graph.list('classGraph') YIELD schemaWithOrientation "
                  "RETURN schemaWithOrientation.nodes AS n")[0]['n']
assert 'pr_mutated' in _sch.get('Compound', {}), \
    'classGraph 투영 안에 pr_mutated 가 없습니다. gds.pageRank.mutate 를 실행했는지 확인하세요'
assert run_cypher(
    "MATCH (n) WHERE 'pr_mutated' IN keys(n) RETURN count(n) AS c")[0]['c'] == 0, \
    '원본에 pr_mutated 가 있습니다. mutate 가 아니라 write 를 쓴 것 같습니다'
print('✅ 통과!')

## 3-5. 저장한 속성으로 Cypher 조회하기
**배경**: 저장해 두면 GDS 없이 평범한 Cypher 로 조건을 얹어 물을 수 있습니다.

**요구사항**:
- `class_rank` 속성이 있는 노드 중 **`PharmacologicClass` 만** 골라 점수 내림차순 1위의 이름을 변수 **`saved_top`** 에 담으세요.

**예시**: `saved_top` 은 **`'Alkylating Activity'`** 입니다. 3-1 의 전체 1위와는 다른 이름입니다 (그쪽은 약물이었습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 이제 GDS 를 부를 필요가 없다. 평범한 MATCH 와 ORDER BY 로 끝난다.

세부구현:
1. 약효분류 레이블로 MATCH 한다.
2. 저장한 속성이 비어 있지 않은 것만 남긴다.
3. 그 속성 내림차순으로 정렬하고 한 건만 받아 이름을 saved_top 에 담는다.
```

</details>

In [ ]:
res = run_cypher('''
MATCH (p:PharmacologicClass)
WHERE p.class_rank IS NOT NULL
RETURN p.name AS name
ORDER BY p.class_rank DESC, name ASC
LIMIT 1
''')
saved_top = res[0]['name']
print(f'saved_top: {saved_top}')


In [ ]:
# [자가채점]
assert saved_top == 'Alkylating Activity', '약효분류 중 1위 이름이 다릅니다. MATCH 를 PharmacologicClass 로 걸었는지, class_rank 내림차순으로 정렬했는지 확인하세요'
# 3-4 가 실제로 저장했어야 조회가 된다. 채점이 같은 조회를 다시 해 대조한다
_saved = run_cypher("MATCH (p:PharmacologicClass) WHERE p.class_rank IS NOT NULL "
                    "RETURN p.name AS name ORDER BY p.class_rank DESC, name LIMIT 1")
assert _saved, 'class_rank 가 저장된 약효분류가 없습니다. 3-4 를 먼저 실행하세요'
assert saved_top == _saved[0]['name'], \
    '지금 다시 조회한 1위와 다릅니다. 저장된 속성으로 직접 조회했는지 확인하세요'
print('✅ 통과!')

---
# 4. 개인화 PageRank 와 투영 정리

출발점을 정한 관점의 순위를 뽑고, 다 쓴 투영을 내립니다(교안_02 3절 · 교안_01 4절).

## 4-1. 개인화 PageRank: 한 약효분류 관점에서 보기
**배경**: 전체 순위 말고 **한 노드 관점**의 순위를 봅니다. `sourceNodes` 에 출발 노드를 넘기면 점수가 그 자리에서 흘러나갑니다.

**요구사항**:
- 약효분류 **`'Vitamin K Inhibitors'`** 를 출발점으로 `classGraph` 에서 개인화 PageRank 를 실행하세요. 출발 분류 이름은 **`$name` 파라미터**로 넘기고, 쿼리 문자열은 **`ppr_query`** 에 담으세요.
- 쿼리의 RETURN 에서 이름 컬럼의 별칭은 **`name`** 으로 두세요. 자가채점 셀이 그 이름으로 값을 읽습니다.
- 점수 내림차순 상위 3개의 **이름 리스트**를 변수 **`ppr_top3`** 에 담으세요.
- 상위 3개는 파이썬에서 자르지 말고 **쿼리 안에서** `ORDER BY score DESC LIMIT 3` 으로 받으세요. 자가채점이 같은 쿼리를 다른 약효분류로 한 번 더 실행해 세 건이 오는지 봅니다.

**예시**: 이 출발점에서는 `ppr_top3[0]` 이 출발점 자신이라 `'Vitamin K Inhibitors'` 입니다. 2·3위는 그 분류에 속한 약 이름이 나옵니다(값은 자가채점에서 확인하세요).

출발점이 늘 1위인 것은 아닙니다. 출발점이 점수를 나눠 줄 이웃이 적으면 그 점수가 몇몇 이웃에게 몰려 이웃이 앞서기도 합니다.

> 채점 셀은 같은 `ppr_query` 를 **다른 약효분류 이름**으로 한 번 더 실행합니다. `ppr_top3` 에 값을 직접 적어 넣으면 여기서 걸립니다.

<details><summary>힌트</summary>

```text
접근방법:
- 출발 노드를 먼저 MATCH 로 찾아 리스트로 모은 뒤 알고리즘 설정에 넘긴다.
- 노드 하나를 리스트로 만드는 집계 함수가 있다.

세부구현:
1. 약효분류 레이블과 이름으로 출발 노드를 찾는다.
2. WITH 절에서 그 노드를 리스트로 모아 변수에 담는다.
3. pageRank stream 을 부르며 설정 맵의 sourceNodes 에 그 리스트를 넘긴다.
4. 그 쿼리 문자열을 ppr_query 에 담고, 출발 분류 이름은 run_cypher 의 name 인자로 넘긴다.
5. 쿼리 안에서 점수 내림차순 3건만 받고, 그 결과에서 이름만 뽑아 파이썬 리스트로 만든다.
```

</details>

In [ ]:
ppr_query = '''
MATCH (p:PharmacologicClass {name: $name})
WITH collect(p) AS sources
CALL gds.pageRank.stream('classGraph', {
    sourceNodes: sources
})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name
ORDER BY score DESC, name ASC
LIMIT 3
'''
res = run_cypher(ppr_query, name='Vitamin K Inhibitors')
ppr_top3 = [row['name'] for row in res]
print(f'ppr_top3: {ppr_top3}')


In [ ]:
# [자가채점]
assert ppr_top3[0] == 'Vitamin K Inhibitors', '이 출발점에서는 1위가 출발 노드 자신입니다. sourceNodes 에 넘긴 노드를 확인하세요'
assert ppr_top3[1] == 'Phenprocoumon', '2위 이름이 다릅니다. classGraph 로 실행했는지, 정렬을 score 내림차순으로 했는지 확인하세요'
assert ppr_top3[2] == 'Warfarin', '3위 이름이 다릅니다. LIMIT 3 으로 세 건을 받았는지 확인하세요'
# 값을 손으로 적어도 통과하지 않게, 같은 쿼리를 다른 약효분류로 한 번 더 돌린다
other = run_cypher(ppr_query, name='Aldosterone Antagonists')
assert other and 'name' in other[0], \
    'ppr_query 의 RETURN 별칭을 name 으로 두세요. 채점 셀이 그 이름으로 읽습니다'
other = [row['name'] for row in other]
assert other == ['Aldosterone Antagonists', 'Spironolactone', 'Eplerenone'], \
    f'같은 쿼리를 다른 분류로 돌리면 그 분류와 거기 속한 약이 나와야 합니다(현재 {other[:5]}). ppr_query 에 쿼리 문자열을 담고 출발 이름을 $name 파라미터로 넘겼는지, LIMIT 3 이 쿼리 안에 있는지 확인하세요'
print('✅ 통과!')

## 4-2. 다 쓴 투영 내리기
**배경**: 투영은 메모리를 차지합니다. 분석이 끝나면 내립니다. 그런데 `gds.graph.list()` 가 준 것을 통째로 지우면 남이 쓰는 중인 투영까지 함께 내려갑니다(교안_01 🧹 절). 그래서 **지울 것을 이름으로** 짚어 냅니다.

이 문항은 3-3 에서 만든 `classDirected` 가 있어야 합니다. 3-3 을 먼저 푸세요.

**요구사항**:
- 역할이 끝난 투영 둘, **`onlyClassGraph`** 와 **`classDirected`** 를 **이름으로 지목해** 삭제하세요. `classGraph` 는 남깁니다.
- 삭제 후 남은 투영 이름 리스트를 변수 **`remain`** 에 담으세요.

**예시**: `remain` 에는 `'classGraph'` 만 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 삭제 프로시저에 이름을 하나씩 넘긴다. 쓸 컬럼만 YIELD 로 골라 받아야 경고가 뜨지 않는다.

세부구현:
1. 삭제 프로시저를 두 번 부른다. onlyClassGraph 와 classDirected 이름을 하나씩 적어 각각 지운다.
2. 2-1 에서 한 것처럼 목록을 다시 받아 이름만 뽑아 remain 에 담는다.
```

</details>

In [ ]:
run_cypher("CALL gds.graph.drop('onlyClassGraph') YIELD graphName")
run_cypher("CALL gds.graph.drop('classDirected') YIELD graphName")
res = run_cypher('CALL gds.graph.list() YIELD graphName RETURN graphName')
remain = [row['graphName'] for row in res]
print(f'remain: {remain}')


In [ ]:
# [자가채점]
# 학생 리스트만 보면 drop 을 안 불러도 통과한다. 카탈로그를 직접 읽어 대조한다
_live = [r['graphName'] for r in run_cypher(
    "CALL gds.graph.list() YIELD graphName RETURN graphName")]
assert 'onlyClassGraph' not in _live, \
    'onlyClassGraph 가 아직 메모리에 있습니다. gds.graph.drop 을 실행했는지 확인하세요'
assert 'classDirected' not in _live, \
    'classDirected 가 아직 메모리에 있습니다. gds.graph.drop 을 실행했는지 확인하세요'
assert 'classGraph' in _live, \
    'classGraph 까지 내려갔습니다. 역할이 끝난 투영 둘만 지목해 내리세요'
assert 'onlyClassGraph' not in remain, 'onlyClassGraph 가 아직 남아 있습니다. drop 을 실행했는지 확인하세요'
assert 'classDirected' not in remain, 'classDirected 가 아직 남아 있습니다. drop 을 실행했는지 확인하세요'
assert 'classGraph' in remain, 'classGraph 까지 지우면 안 됩니다. 이름을 확인하세요'
print('✅ 통과!')

---
## 🧹 다 쓴 투영 내리기
4-2 에서 남겨 둔 `classGraph` 도 과제가 끝났으니 내립니다. 투영은 노트북이 아니라 **Neo4j 서버 메모리**에 있어 노트북을 닫아도 사라지지 않습니다.

In [ ]:
# [제공 코드] 이 과제에서 만든 투영을 내립니다: 이 셀은 실행만 하세요.
# 이름을 하나씩 적습니다. 두 번째 인자 false 는 그 이름이 없으면 그냥 넘어가라는 뜻입니다
for _name in ['classGraph', 'classDirected', 'onlyClassGraph']:
    run_cypher("CALL gds.graph.drop($name, false) YIELD graphName RETURN graphName",
               name=_name)
print('남은 투영:', [row['graphName'] for row in
      run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName")])

---
수고했어요! 모든 자가채점이 **✅ 통과!** 로 끝났는지 확인하세요. 3-1 과 3-2 의 답이 왜 달랐는지 한 문장으로 말할 수 있으면 이 단원의 절반은 이해한 것입니다.